# QuestMasters — DM Orchestrator en Colab

Corre el modelo Gemma 4 26B-A4B-it (4-bit) con LoRAs en la GPU de Colab.
Requiere GPU con 15GB+ VRAM (T4 ajustado, A100 ideal).
Tu backend local se conecta a este servidor via ngrok.

**Pasos:**
1. Configura los secretos en el panel izquierdo (icono de llave):
   - `HF_TOKEN` — tu token de HuggingFace
   - `NGROK_TOKEN` — tu token de ngrok (gratis en https://ngrok.com)
2. Ejecuta todas las celdas
3. Copia la URL de ngrok que aparece al final
4. En tu `.dev.vars` del backend, pon:
   ```
   DM_MODEL_ENDPOINT_MAS=<url_ngrok>
   DM_MODEL_ENDPOINT_MONOLITHIC=<url_ngrok>
   ```
5. Reinicia el backend (`bun dev`)

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes safetensors \
    huggingface_hub fastapi uvicorn pyngrok lightrag-hku langgraph \
    networkx chromadb pydantic numpy

In [ ]:
import os
from google.colab import userdata

# ── Secretos de Colab (configuralos en el icono de llave del panel izquierdo) ──
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["DM_MODE"] = "local"
os.environ["DM_LOCAL_PORT"] = "8000"
NGROK_TOKEN = userdata.get("NGROK_TOKEN")

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

## 1. Clonar el orchestrator desde el repo

In [ ]:
%%bash
if [ ! -d "/content/orchestrator" ]; then
  mkdir -p /content/orchestrator
fi

In [ ]:
%%writefile /content/orchestrator/config.py
import os
from pathlib import Path

MODE = "local"
HF_TOKEN = os.environ.get("HF_TOKEN", "")
BASE_MODEL_ID = "google/gemma-4-26B-A4B-it"
BASE_MODEL_LOCAL = Path("/content/base_model")
LORA_BASE_PATH = Path("/content/lora_weights")
SESSIONS_DIR = Path("/content/sessions")
LORA_REPOS = {
    "narrator": "Questmasters/lora_narrador",
    "arbiter": "Questmasters/lora_reglas",
    "npc": "Questmasters/lora_npc",
    "state": "Questmasters/lora_estado",
    "monolithic": "Questmasters/qlora_monolitico",
}

## 2. Cargar modelo Gemma 4 26B-A4B-it + LoRAs (4-bit NF4)

In [ ]:
import torch
import gc
from huggingface_hub import snapshot_download
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from pathlib import Path

HF_TOKEN = os.environ["HF_TOKEN"]
BASE_MODEL_ID = "google/gemma-4-26B-A4B-it"

# En modelos MoE (26B-A4B), PEFT solo permite UN LoRA con target_parameters a la vez.
ACTIVE_LORA_NAME = "monolithic"
ACTIVE_LORA_REPO = "Questmasters/qlora_monolitico"
LORA_BASE_PATH = Path("/content/lora_weights")

# Descargar el LoRA activo
lora_dir = LORA_BASE_PATH / ACTIVE_LORA_NAME
if not lora_dir.exists():
    print(f"Descargando LoRA '{ACTIVE_LORA_NAME}' desde {ACTIVE_LORA_REPO}...")
    snapshot_download(repo_id=ACTIVE_LORA_REPO, local_dir=str(lora_dir), token=HF_TOKEN)
else:
    print(f"LoRA '{ACTIVE_LORA_NAME}' en cache")

# Info GPU
print(f"\nCargando {BASE_MODEL_ID} (4-bit NF4)...")
print(f"GPU: {torch.cuda.get_device_name(0)}")
free, total = torch.cuda.mem_get_info(0)
free_gb = free / 1024**3
total_gb = total / 1024**3
print(f"VRAM disponible: {free_gb:.1f} GB / {total_gb:.1f} GB")

# Limpiar cache antes de cargar
gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# device_map="auto" permite offload a CPU durante la cuantización
# Una vez cargado, el modelo cuantizado cabe en GPU
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN)
print("Modelo base cargado.")

# Cargar LoRA
print(f"\nCargando LoRA '{ACTIVE_LORA_NAME}'...")
model = PeftModel.from_pretrained(base, str(lora_dir), adapter_name=ACTIVE_LORA_NAME)
print(f"LoRA '{ACTIVE_LORA_NAME}' cargado.")

print("\nModelo listo!")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f"VRAM usada: {(total - free) / 1024**3:.1f} GB / {total / 1024**3:.1f} GB")
    print(f"VRAM libre: {free / 1024**3:.1f} GB")

## 3. Levantar servidor FastAPI + ngrok

In [ ]:
%%writefile /content/orchestrator/schemas.py
from __future__ import annotations
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field


class CharacterSnapshot(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    name: str
    race: str = ""
    class_name: str = Field(default="", alias="class")
    background: str = ""
    description: str = ""


class DmModelRequest(BaseModel):
    session_id: str
    architecture_type: Literal["monolithic", "mas"]
    model_id: str
    campaign_prompt: str
    characters: list[CharacterSnapshot]
    conversation_history: list[dict]
    player_input: str | None = None
    current_memory_snapshot: dict


class DeltaChunk(BaseModel):
    type: Literal["delta"] = "delta"
    content: str


class MemorySnapshot(BaseModel):
    l2_episode_ids: list[str] = []
    l3_entity_ids: list[str] = []


class UsageStats(BaseModel):
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int


class MetadataChunk(BaseModel):
    type: Literal["metadata"] = "metadata"
    memory_snapshot: MemorySnapshot
    usage: UsageStats
    latency_ms: int


class DoneChunk(BaseModel):
    type: Literal["done"] = "done"


class ErrorChunk(BaseModel):
    type: Literal["error"] = "error"
    message: str


SseChunk = DeltaChunk | MetadataChunk | DoneChunk | ErrorChunk

In [ ]:
import asyncio
import json
import re
import time
import queue
import threading
import logging
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Any, AsyncGenerator

!pip install -q nest_asyncio
import nest_asyncio
nest_asyncio.apply()

import numpy as np
import torch
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from transformers import AutoModel, AutoTokenizer as EmbedTokenizer, TextIteratorStreamer

import sys
sys.path.insert(0, "/content/orchestrator")
from schemas import (
    DeltaChunk, DmModelRequest, DoneChunk, ErrorChunk,
    MetadataChunk, MemorySnapshot, UsageStats, SseChunk,
)

log = logging.getLogger("dm-orchestrator")
app = FastAPI(title="QuestMasters DM Orchestrator (Colab)")
_executor = ThreadPoolExecutor(max_workers=1)

_CAMEL_RE_1 = re.compile(r"(.)([A-Z][a-z]+)")
_CAMEL_RE_2 = re.compile(r"([a-z0-9])([A-Z])")

# ── Config ──
USE_BASE_MODEL_ONLY = True
SESSIONS_DIR = Path("/content/sessions")
SESSIONS_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════
# LightRAG — memoria de largo plazo (NPCs, lugares, eventos)
# ═══════════════════════════════════════════════════════════════════════

_EMBED_MODEL_ID = "BAAI/bge-small-en-v1.5"
_EMBED_DIM = 384
_embed_tok = None
_embed_mod = None

def _load_embed_model():
    global _embed_tok, _embed_mod
    if _embed_mod is None:
        print("[rag] Cargando modelo de embeddings (CPU)...")
        _embed_tok = EmbedTokenizer.from_pretrained(_EMBED_MODEL_ID)
        _embed_mod = AutoModel.from_pretrained(_EMBED_MODEL_ID)
        _embed_mod.eval()
        print("[rag] Modelo de embeddings listo.")
    return _embed_tok, _embed_mod

_load_embed_model()


async def _embedding_func(texts: list[str]) -> np.ndarray:
    etok, emod = _embed_tok, _embed_mod
    inputs = etok(texts, return_tensors="pt", truncation=True, max_length=512, padding=True)
    with torch.no_grad():
        out = emod(**inputs)
    return out.last_hidden_state[:, 0, :].cpu().float().numpy()


_async_executor = ThreadPoolExecutor(max_workers=1)

def _run_async(coro):
    future = _async_executor.submit(asyncio.run, coro)
    return future.result()


async def _llm_func(prompt: str, **_kwargs) -> str:
    """LightRAG usa esto internamente para extraer entidades del texto."""
    if USE_BASE_MODEL_ONLY and hasattr(model, "disable_adapter_layers"):
        model.disable_adapter_layers()
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True, return_dict=False,
    ).to(model.device)
    output_ids = model.generate(input_ids=input_ids, max_new_tokens=256, do_sample=False)
    return tokenizer.decode(output_ids[0][input_ids.shape[1]:], skip_special_tokens=True)


def _lightrag_working_dir(session_id: str) -> Path:
    path = SESSIONS_DIR / session_id / "lightrag"
    path.mkdir(parents=True, exist_ok=True)
    return path


def _get_lightrag(session_id: str):
    try:
        from lightrag import LightRAG
        from lightrag.utils import EmbeddingFunc

        embed = EmbeddingFunc(embedding_dim=_EMBED_DIM, max_token_size=512, func=_embedding_func)
        rag = LightRAG(
            working_dir=str(_lightrag_working_dir(session_id)),
            llm_model_func=_llm_func,
            embedding_func=embed,
        )
        _run_async(rag.initialize_storages())
        return rag
    except Exception as exc:
        print(f"[rag] LightRAG no disponible: {exc}")
        return None


def _retrieve_context(session_id: str, query: str) -> str:
    working_dir = _lightrag_working_dir(session_id)
    if not any(working_dir.iterdir()):
        return ""
    rag = _get_lightrag(session_id)
    if rag is None:
        return ""
    try:
        from lightrag import QueryParam
        result = _run_async(rag.aquery(query, param=QueryParam(mode="hybrid")))
        if result:
            print(f"[rag] Contexto recuperado ({len(result)} chars)")
        return result
    except Exception as exc:
        print(f"[rag] Error en query: {exc}")
        return ""


def _insert_turn(session_id: str, turn_number: int, player_input: str, dm_response: str, characters: list) -> None:
    rag = _get_lightrag(session_id)
    if rag is None:
        return
    try:
        char_names = ", ".join(c.name for c in characters) if characters else "Jugador"
        text = (
            f"--- Turno {turn_number} ---\n"
            f"Personaje jugador: {char_names}\n"
        )
        if player_input:
            text += f"Acción del jugador: {player_input}\n"
        text += f"Narración del DM: {dm_response}\n"
        _run_async(rag.ainsert(text))
        print(f"[rag] Turno {turn_number} insertado en memoria")
    except Exception as exc:
        print(f"[rag] Error al insertar turno: {exc}")


# ═══════════════════════════════════════════════════════════════════════
# Prompt y generación
# ═══════════════════════════════════════════════════════════════════════

def _camel_to_snake(name: str) -> str:
    return _CAMEL_RE_2.sub(r"\1_\2", _CAMEL_RE_1.sub(r"\1_\2", name)).lower()

def _convert_keys(obj):
    if isinstance(obj, dict):
        return {_camel_to_snake(k): _convert_keys(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_convert_keys(i) for i in obj]
    return obj


def _is_opening_turn(request: DmModelRequest) -> bool:
    return not request.player_input and len(request.conversation_history) == 0


_SYSTEM_BASE = """\
Eres un Dungeon Master de D&D 5e. Responde en español.

REGLAS:
- Máximo 3 párrafos cortos por respuesta.
- Oraciones cortas. Máximo 20 palabras por oración.
- NO inventes palabras raras ni nombres absurdos.
- Usa "tú" para dirigirte al jugador. NO uses "vosotros".
- Solo hay los personajes listados abajo. NO inventes más.
- Sé concreto: describe lo que el personaje ve, oye y huele.
- Cuando crees un PNJ, dale un nombre propio y una descripción breve memorable.
- Mantén consistencia con PNJs y lugares ya establecidos.
- Termina con una pregunta clara: "¿Qué haces?" o similar."""

_OPENING_EXTRA = """
ESCENA INICIAL:
1. Describe el lugar en 2-3 oraciones cortas.
2. Describe por qué el personaje está ahí en 1-2 oraciones.
3. Presenta UN evento o persona que llame la atención. Dale nombre y descripción.
4. Pregunta qué hace el jugador."""

_TURN_EXTRA = """
- Responde directamente a lo que el jugador dijo.
- Describe lo que pasa en 2-3 oraciones.
- Si hay PNJs presentes, úsalos por su nombre.
- Pregunta qué hace el jugador."""


def _build_messages(request: DmModelRequest, rag_context: str = "") -> list[dict]:
    chars = []
    for c in request.characters:
        parts = [c.name]
        if c.race:
            parts.append(c.race)
        if c.class_name:
            parts.append(c.class_name)
        chars.append(", ".join(parts))
    chars_block = "\n".join(f"- {c}" for c in chars) if chars else "- Sin personajes"

    opening = _is_opening_turn(request)
    extra = _OPENING_EXTRA if opening else _TURN_EXTRA

    system = (
        f"{_SYSTEM_BASE}\n{extra}\n\n"
        f"Campaña: {request.campaign_prompt}\n\n"
        f"Personajes:\n{chars_block}"
    )
    if rag_context:
        system += f"\n\nContexto de turnos anteriores (PNJs, lugares, eventos):\n{rag_context}"

    messages = [{"role": "system", "content": system}]

    for turn in request.conversation_history[-6:]:
        role = "user" if turn.get("role") == "player" else "assistant"
        messages.append({"role": role, "content": turn["content"]})

    if request.player_input:
        messages.append({"role": "user", "content": request.player_input})

    return messages


def _clean_response(text: str) -> str:
    import unicodedata
    cleaned = []
    for char in text:
        cat = unicodedata.category(char)
        if cat.startswith("So") or cat.startswith("Cs"):
            continue
        if ord(char) > 0x2000 and not char in "—–''\"\"…·":
            continue
        cleaned.append(char)
    return "".join(cleaned)


_turn_counters: dict[str, int] = {}

def _generate_to_queue(request: DmModelRequest, q: queue.Queue) -> None:
    import traceback
    try:
        if USE_BASE_MODEL_ONLY:
            model.disable_adapter_layers()
        else:
            model.enable_adapter_layers()
            adapter = "monolithic" if request.architecture_type == "monolithic" else "narrator"
            model.set_adapter(adapter)

        opening = _is_opening_turn(request)
        mode = "BASE" if USE_BASE_MODEL_ONLY else "LoRA"
        print(f"[gen] {mode} | {'apertura' if opening else 'turno'}")

        rag_context = ""
        if not opening and request.player_input:
            rag_context = _retrieve_context(request.session_id, request.player_input)

        messages = _build_messages(request, rag_context)
        input_ids = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True,
            return_dict=False,
        ).to(model.device)

        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

        gen_kwargs = {
            "input_ids": input_ids,
            "max_new_tokens": 400 if opening else 300,
            "temperature": 0.7,
            "top_k": 40,
            "top_p": 0.9,
            "do_sample": True,
            "streamer": streamer,
            "repetition_penalty": 1.3,
        }
        t = threading.Thread(target=model.generate, kwargs=gen_kwargs, daemon=True)
        t.start()

        t_start = time.monotonic()
        full_response = ""
        for token in streamer:
            cleaned = _clean_response(token)
            if cleaned:
                full_response += cleaned
                q.put(DeltaChunk(content=cleaned))

        latency_ms = int((time.monotonic() - t_start) * 1000)

        turn_num = _turn_counters.get(request.session_id, 0) + 1
        _turn_counters[request.session_id] = turn_num
        threading.Thread(
            target=_insert_turn,
            args=(request.session_id, turn_num, request.player_input or "", full_response, request.characters),
            daemon=True,
        ).start()

        q.put(MetadataChunk(
            memory_snapshot=MemorySnapshot(),
            usage=UsageStats(prompt_tokens=int(input_ids.shape[1]), completion_tokens=0, total_tokens=0),
            latency_ms=latency_ms,
        ))
        q.put(DoneChunk())
        q.put(None)
    except Exception as exc:
        exc.__traceback_str__ = traceback.format_exc()
        q.put(exc)


def _chunk_to_sse(chunk: SseChunk, model_id: str) -> str:
    if isinstance(chunk, DeltaChunk):
        payload = {"type": "delta", "delta": chunk.content}
    elif isinstance(chunk, MetadataChunk):
        payload = {
            "type": "metadata",
            "metadata": {
                "memorySnapshot": chunk.memory_snapshot.model_dump(),
                "narrativeNotesDelta": [],
                "usage": {
                    "inputTokens": chunk.usage.prompt_tokens,
                    "outputTokens": chunk.usage.completion_tokens,
                },
                "latencyMs": chunk.latency_ms,
                "modelId": model_id,
            },
        }
    elif isinstance(chunk, DoneChunk):
        payload = {"type": "done"}
    elif isinstance(chunk, ErrorChunk):
        payload = {"type": "error", "error": chunk.message}
    else:
        return ""
    return f"data: {json.dumps(payload)}\n\n"


@app.get("/health")
async def health():
    return {"status": "ok", "mode": "colab-26b", "gpu": torch.cuda.get_device_name(0), "base_only": USE_BASE_MODEL_ONLY, "lightrag": True}


@app.post("/generate")
async def generate(body: dict[str, Any]) -> StreamingResponse:
    session_id = body.get("sessionId", body.get("session_id", "?"))
    mode = "BASE" if USE_BASE_MODEL_ONLY else "LoRA"
    print(f"\n{'='*40}")
    print(f"Request | session={session_id} | mode={mode}")

    try:
        snake_body = _convert_keys(body)
        request = DmModelRequest(**snake_body)
    except Exception as exc:
        error = json.dumps({"type": "error", "error": f"Input invalido: {exc}"})
        return StreamingResponse(iter([f"data: {error}\n\n"]), media_type="text/event-stream")

    chunk_queue: queue.Queue = queue.Queue()
    loop = asyncio.get_event_loop()
    loop.run_in_executor(_executor, _generate_to_queue, request, chunk_queue)

    async def stream() -> AsyncGenerator[str, None]:
        while True:
            try:
                item = chunk_queue.get_nowait()
            except queue.Empty:
                yield ": keepalive\n\n"
                await asyncio.sleep(2)
                continue
            if item is None:
                break
            if isinstance(item, Exception):
                tb = getattr(item, "__traceback_str__", str(item))
                print(f"Error:\n{tb}")
                yield f"data: {json.dumps({'type': 'error', 'error': str(item)})}\n\n"
                break
            yield _chunk_to_sse(item, request.model_id)

    return StreamingResponse(stream(), media_type="text/event-stream")

print(f"FastAPI app definida. Mode: {'BASE (sin LoRA)' if USE_BASE_MODEL_ONLY else 'LoRA activo'} | LightRAG: ON")

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_TOKEN)

tunnel = ngrok.connect(8000)
public_url = tunnel.public_url
print("=" * 60)
print(f"SERVIDOR PUBLICO: {public_url}")
print("=" * 60)
print(f"\nPon esto en tu .dev.vars del backend:")
print(f"  DM_MODEL_ENDPOINT_MAS={public_url}")
print(f"  DM_MODEL_ENDPOINT_MONOLITHIC={public_url}")
print(f"  DM_USE_RUNPOD=false")

In [ ]:
# Iniciar servidor (esta celda bloquea - el servidor queda corriendo)
import uvicorn

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()